# Trabalho 2 — Apache Spark com MinIO e PostgreSQL
## Pipeline: PostgreSQL → landing-zone (CSV) → bronze (Delta Lake)

**Etapas:**
1. Configurar SparkSession com Delta Lake + S3/MinIO
2. Extrair tabelas do PostgreSQL e gravar na `landing-zone` em CSV
3. Ler os CSVs da `landing-zone` e gravar em Delta Lake no bucket `bronze`
4. Executar operações DML (INSERT, UPDATE, DELETE) nas tabelas Delta
5. Demonstrar Time Travel

## 1. Imports e configurações

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from delta import *

# ============================================================
# CONFIGURAÇÕES — ajuste se necessário
# ============================================================
MINIO_ENDPOINT   = "http://localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"

PG_HOST     = "localhost"
PG_PORT     = "5432"
PG_DB       = "loja_db"
PG_USER     = "admin"
PG_PASSWORD = "admin123"
PG_URL      = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"

LANDING_ZONE = "s3a://landing-zone"
BRONZE       = "s3a://bronze"

TABELAS = ["produtos", "clientes", "vendas"]

## 2. SparkSession com Delta Lake e MinIO (S3A)

In [2]:
spark = (
    SparkSession.builder
    .appName("Trabalho2-SparkMinIO")
    # Delta Lake
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.2.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.3")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint",              MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",            MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",            MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",     "true")
    .config("spark.hadoop.fs.s3a.impl",                  "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled","false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession criada com sucesso!")
spark

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## 3. Extração: PostgreSQL → landing-zone (CSV)

In [ ]:
pg_props = {
    "user":     PG_USER,
    "password": PG_PASSWORD,
    "driver":   "org.postgresql.Driver"
}

for tabela in TABELAS:
    print(f"\n📥 Extraindo tabela: {tabela}")
    df = spark.read.jdbc(url=PG_URL, table=tabela, properties=pg_props)
    df.printSchema()
    df.show(5)

    destino = f"{LANDING_ZONE}/{tabela}"
    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(destino)

    print(f"✅ Tabela '{tabela}' gravada em: {destino}")

print("\n🎉 Extração concluída! Todas as tabelas estão na landing-zone.")

## 4. Transformação: landing-zone (CSV) → bronze (Delta Lake)

In [ ]:
for tabela in TABELAS:
    print(f"\n🔄 Convertendo '{tabela}' para Delta Lake...")

    origem  = f"{LANDING_ZONE}/{tabela}"
    destino = f"{BRONZE}/{tabela}"

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(origem)

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(destino)

    # Registra como tabela SQL
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}_bronze
        USING DELTA
        LOCATION '{destino}'
    """)

    count = spark.read.format("delta").load(destino).count()
    print(f"✅ '{tabela}_bronze' criada com {count} registros em: {destino}")

print("\n🎉 Todas as tabelas estão no bucket bronze em formato Delta Lake!")

## 5. DML — INSERT

In [ ]:
print("=" * 50)
print("ESTADO ATUAL DA TABELA vendas_bronze")
print("=" * 50)
spark.sql("SELECT * FROM vendas_bronze ORDER BY id_venda").show()

# INSERT — novos registros de venda
spark.sql("""
INSERT INTO vendas_bronze VALUES
(100, 1, 3, 2, 120.00, '2024-03-01'),
(101, 2, 5, 1, 800.00, '2024-03-02')
""")

print("\n✅ INSERT executado!")
print("\nTabela APÓS o INSERT:")
spark.sql("SELECT * FROM vendas_bronze ORDER BY id_venda").show()

## 6. DML — UPDATE

In [ ]:
# UPDATE — corrige o valor_unitario da venda 100
spark.sql("""
UPDATE vendas_bronze
SET valor_unitario = 999.99
WHERE id_venda = 100
""")

print("✅ UPDATE executado! Novo valor da venda 100:")
spark.sql("SELECT * FROM vendas_bronze WHERE id_venda = 100").show()

## 7. DML — DELETE

In [ ]:
# DELETE — remove a venda 101
spark.sql("""
DELETE FROM vendas_bronze
WHERE id_venda = 101
""")

print("✅ DELETE executado!")
print("\nRegistros com id_venda >= 100 após DELETE:")
spark.sql("SELECT * FROM vendas_bronze WHERE id_venda >= 100").show()

## 8. Time Travel — histórico de versões

In [ ]:
vendas_path = f"{BRONZE}/vendas"

print("📜 Histórico de versões da tabela vendas:")
spark.sql("DESCRIBE HISTORY vendas_bronze") \
     .select("version", "timestamp", "operation") \
     .show(truncate=False)

In [ ]:
# Leitura da versão 0 (estado original após ingestão)
print("🕐 Versão 0 — estado logo após a ingestão do CSV:")
spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .load(vendas_path) \
    .show()

## 9. Tabelas Gerenciadas vs Não Gerenciadas

| Aspecto | Gerenciada | Não Gerenciada (External) |
|---|---|---|
| Dados | Spark controla localização | Você define o path |
| DROP TABLE | Remove dados e metadados | Remove só metadados |
| Localização padrão | warehouse Spark | Qualquer path (ex: MinIO) |
| Uso típico | Tabelas temporárias | Data Lake / Produção |

Neste trabalho usamos tabelas **não gerenciadas** (external), pois os dados ficam no MinIO e registramos apenas os metadados no Spark com `LOCATION`.

In [ ]:
# Verificando o tipo das tabelas criadas
spark.sql("DESCRIBE EXTENDED vendas_bronze") \
     .filter("col_name IN ('Type', 'Location', 'Provider')") \
     .show(truncate=False)